## Imports

In [105]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

In [106]:
from funcoes_escoragem import *

## Diretório

In [107]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [108]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [109]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 12 WEEK)
    AND DATE(data) < CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,6057130901,4123856,2026-05-12,2026-05-12,1,NI,3562.000000000,BLEND3_3,4123856,"{""manual"":false,""node"":null,""pessoas"":[{""nome""...",...,0,1778785248777,"{""valor_aluguel"":""6950"",""imobiliaria_id"":56325...",2026-05-14 19:00:51+00:00,1778785250479585833,06057130901,B,2026-05-14 19:01:45.644000+00:00,2026-05-12 22:04:16+00:00,DERIVAR
1,10843678747,4351315,2026-07-08,2026-07-08,1,NI,0E-9,,4351315,"{""pessoas"":[{""nome"":"""",""documento"":""1084367874...",...,15,1778785248777,"{""valor_aluguel"":1850,""matchmaking_on"":false,""...",2026-07-09 08:00:51+00:00,1783584050015360220,10843678747,N,2026-07-09 08:03:13.870000+00:00,2026-07-08 20:05:39+00:00,DERIVAR
2,11119112311,4135339,2026-05-15,2026-05-15,1,C,1164.500000000,BLEND3_3,4135339,"{""manual"":false,""node"":null,""pessoas"":[{""nome""...",...,15,1778785248777,"{""valor_aluguel"":""2090"",""imobiliaria_id"":7455,...",2026-05-15 18:00:45+00:00,1778868041183775084,11119112311,N,2026-05-15 18:09:38.393000+00:00,2026-05-15 11:27:06+00:00,DERIVAR
3,94674183120,4190447,2026-05-29,2026-05-29,1,C,0E-9,BVS_CUSTOM,4190447,"{""pessoas"":[{""nome"":"""",""documento"":""9467418312...",...,28,1778785248777,"{""valor_aluguel"":""1500"",""imobiliaria_id"":14563...",2026-05-29 18:00:33+00:00,1780077627902471624,94674183120,E,2026-05-29 18:09:37.889000+00:00,2026-05-29 09:33:05+00:00,REPROVAR
4,48303377884,4284443,2026-06-22,2026-06-22,1,E,0E-9,BLEND3_3,4284443,"{""pessoas"":[{""nome"":"""",""documento"":"""",""classif...",...,28,1778785248777,"{""valor_aluguel"":""790"",""imobiliaria_id"":17939,...",2026-06-23 08:00:35+00:00,1782201633850189970,48303377884,E,2026-06-23 08:12:37.815000+00:00,2026-06-22 18:16:28+00:00,REPROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327706,67773486620,4323673,2026-07-02,2026-07-02,1,NI,1301.500000000,BLEND3_3,4323673,"{""pessoas"":[{""nome"":""MARIA JOSE DE JESUS GONCA...",...,37,1778785248777,"{""valor_aluguel"":1050,""matchmaking_on"":false,""...",2026-07-02 18:00:45+00:00,1783015241711278717,67773486620,E,2026-07-02 18:07:44.317000+00:00,2026-07-02 11:28:32+00:00,REPROVAR
327707,10753086824,4356912,2026-07-10,2026-07-10,3,NI,13289.000000000,HVA3,4356912,"{""pessoas"":[{""nome"":""MARIA NEIDA JULIA ANTUNES...",...,37,1778785248777,"{""valor_aluguel"":2500,""matchmaking_on"":false,""...",2026-07-10 18:00:43+00:00,1783706441557822363,10753086824,E,2026-07-10 18:08:41.681000+00:00,2026-07-10 14:07:39+00:00,REPROVAR
327708,23555432672,4372238,2026-07-14,2026-07-14,1,A,3699.000000000,BLEND3_3,4372238,"{""pessoas"":[{""nome"":""DARCI VIEIRA CALDAS"",""doc...",...,37,1778785248777,"{""valor_aluguel"":1300,""matchmaking_on"":false,""...",2026-07-15 08:00:37+00:00,1784102433864195583,23555432672,C,2026-07-15 08:04:36.654000+00:00,2026-07-14 16:16:18+00:00,REPROVAR
327709,38470659049,4067108,2026-04-29,2026-04-29,1,NI,4247.000000000,BLEND3_3,4067108,"{""manual"":false,""node"":null,""pessoas"":[{""nome""...",...,37,1778785248777,None,2026-05-20 08:01:14+00:00,1779264072845223404,38470659049,E,2026-05-20 08:09:08.654000+00:00,2026-04-29 08:03:52+00:00,DERIVAR


In [110]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [111]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                278178
BLEND_REGRESSAO_2026     21632
HVA3                     12767
BLEND_4                   8550
BVS_CUSTOM                4370
HVA4                      1907
BVS_CUSTOM_V2              152
HFT1                       111
                            39
None                         4
Name: count, dtype: int64

## Multiproponente

In [112]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    315127
2     11553
3       885
4       133
5         7
6         3
7         1
8         1
Name: count, dtype: Int64

In [113]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.961603
2    0.035254
3    0.002701
4    0.000406
5    0.000021
6    0.000009
7    0.000003
8    0.000003
Name: proportion, dtype: Float64

## Quebrar JSON - Retornado

In [114]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [115]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

# json_normalize resets the index; preserve credpago_df labels for the join below
contrato_df = pd.json_normalize(payload_parsed.tolist(), sep="_")
contrato_df.index = payload_parsed.index
contrato_df = contrato_df.add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.

pessoa_records = payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0])
pessoa_df = pd.json_normalize(pessoa_records.tolist(), sep="_")
pessoa_df.index = payload_parsed.index
pessoa_df = pessoa_df.add_prefix("pessoa_")

In [116]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [117]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_valid = request_parsed.dropna()
request_df = pd.json_normalize(request_valid.tolist(), sep="_")
request_df.index = request_valid.index
request_df = request_df.add_prefix("request_")

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [118]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [119]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [120]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts
471,0.000000,0.000000
817,0.000000,0.109366
1265,NaN,0.130864
1444,0.163121,0.131915
1700,0.000000,0.060096
...,...,...
325458,NaN,0.109366
325534,NaN,NaN
325891,0.000000,0.077220
326467,0.195122,0.117647


## Escoragem Blend4

In [121]:
credpago_clean_rep = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [122]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [123]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [124]:
df_predict = (credpago_clean_rep.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] <= 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_realEstatePc4HistoryOver100Contracts,request_pessoa_principal_nome,request_scores_HFT1,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,6057130901,4123856,2026-05-12,2026-05-12,1,NI,BLEND3_3,5787216,0,06057130901,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,3562.0
1,10843678747,4351315,2026-07-08,2026-07-08,1,NI,,6070184,15,10843678747,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,0.0
2,11119112311,4135339,2026-05-15,2026-05-15,1,C,BLEND3_3,5801628,15,11119112311,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1164.5
3,94674183120,4190447,2026-05-29,2026-05-29,1,C,BVS_CUSTOM,5870560,28,94674183120,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,0.0
4,48303377884,4284443,2026-06-22,2026-06-22,1,E,BLEND3_3,5986971,28,48303377884,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,0.0


In [125]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    326787
E_BVS        923
dtype: int64

In [126]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [127]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [128]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [129]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [130]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada.copy()
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,6057130901,4123856,2026-05-12,2026-05-12,1,NI,BLEND3_3,5787216,0,06057130901,...,0.0,0.30,0.0,3.360187,0.000000,-0.835486,0.546905,453.0,B,E
1,10843678747,4351315,2026-07-08,2026-07-08,1,NI,,6070184,15,10843678747,...,0.0,0.00,0.0,0.000000,0.000000,0.000000,0.490743,509.0,A,None
2,11119112311,4135339,2026-05-15,2026-05-15,1,C,BLEND3_3,5801628,15,11119112311,...,0.0,-1.70,0.0,3.014739,0.004918,-0.008522,0.615218,385.0,D,None
3,94674183120,4190447,2026-05-29,2026-05-29,1,C,BVS_CUSTOM,5870560,28,94674183120,...,0.0,0.00,0.0,0.000000,0.000000,-0.644695,0.488512,511.0,A,None
4,48303377884,4284443,2026-06-22,2026-06-22,1,E,BLEND3_3,5986971,28,48303377884,...,0.0,-1.70,0.0,0.000000,0.000000,0.174016,0.518551,481.0,A,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327705,67773486620,4323673,2026-07-02,2026-07-02,1,NI,BLEND3_3,6035578,37,67773486620,...,0.0,2.45,0.0,0.832349,0.000000,-0.689886,0.504345,496.0,A,E
327706,10753086824,4356912,2026-07-10,2026-07-10,3,NI,HVA3,6078096,37,10753086824,...,0.0,2.55,6.0,-0.534154,0.000000,0.000000,0.521048,479.0,B,None
327707,23555432672,4372238,2026-07-14,2026-07-14,1,A,BLEND3_3,6095866,37,23555432672,...,0.0,2.45,1.0,-0.173395,0.000000,-0.555452,0.493116,507.0,A,B
327708,38470659049,4067108,2026-04-29,2026-04-29,1,NI,BLEND3_3,5724332,37,38470659049,...,0.0,1.35,0.0,0.000000,0.000000,-0.801651,0.489358,511.0,A,E


In [131]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                            43
BLEND3_3                278178
BLEND_4                   8550
BLEND_REGRESSAO_2026     21632
BVS_CUSTOM                4370
BVS_CUSTOM_V2              152
HFT1                       111
HVA3                     12767
HVA4                      1907
dtype: int64

In [132]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [133]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    326787
E_BVS        923
dtype: int64

In [134]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                            43
BLEND3_3                278178
BLEND_4                   8550
BLEND_REGRESSAO_2026     21632
BVS_CUSTOM                4370
BVS_CUSTOM_V2              152
HFT1                       111
HVA3                     12767
HVA4                      1907
dtype: int64

In [135]:
cp_final_df.groupby("model", dropna=False).size()

model
                            39
BLEND3_3                278178
BLEND_4                   8550
BLEND_REGRESSAO_2026     21632
BVS_CUSTOM                4370
BVS_CUSTOM_V2              152
HFT1                       111
HVA3                     12767
HVA4                      1907
NaN                          4
dtype: int64

In [136]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    326787
E_BVS        923
dtype: int64

In [137]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)